# Kenya County Mental Health Risk Baseline

This notebook connects `khis-toolkit` to Open Health Risk Engine so county-level KHIS signals can be scored with the current NHANES baseline model.

> This uses a model trained on NHANES (US) data as a baseline. Phase 3 will retrain on Kenya-specific clinical data once KHIS credentials and MOH partnership are established.

In [ ]:
from pathlib import Path
import sys

CURRENT = Path.cwd().resolve()
OHRE_ROOT = CURRENT if (CURRENT / "src").exists() else CURRENT.parent
KHIS_ROOT = OHRE_ROOT.parent / "khis-toolkit"

if str(OHRE_ROOT) not in sys.path:
    sys.path.insert(0, str(OHRE_ROOT))
if KHIS_ROOT.exists() and str(KHIS_ROOT) not in sys.path:
    sys.path.append(str(KHIS_ROOT))


In [ ]:
import pandas as pd
from IPython.display import display

import khis
from dashboard.map import create_county_map
from src.khis_integration import load_khis_mental_health, score_county_risk

In [ ]:
conn = khis.connect()
status, message = conn.ping()
print(message)

counties = khis.list_counties()["name"].tolist()
print(f"Scoring {len(counties)} counties")

In [ ]:
proxy_df = load_khis_mental_health(conn, counties)
display(
    proxy_df[[
        "county",
        "profile_name",
        "mns_burden_index",
        "service_pressure_index",
        "age",
        "poverty_ratio",
        "met_min_week",
        "sleep_hours",
        "general_health",
    ]].head(12)
)

In [ ]:
county_summary = score_county_risk(proxy_df)
display(
    county_summary[[
        "county",
        "mean_risk_score",
        "pct_high_risk",
        "mean_phq9_estimate",
        "mns_burden_index",
        "risk_label",
    ]].head(15)
)

In [ ]:
risk_map = create_county_map(
    county_summary[["county", "mean_risk_score"]].rename(columns={"mean_risk_score": "risk_score"}),
    value_col="risk_score",
    title="Kenya County Mental Health Risk Baseline",
)
risk_map

## Interpretation

- Higher scores here mean counties have a stronger aggregate MNS burden signal after mapping county service activity into a small synthetic cohort for baseline scoring.
- This is useful for prioritising where to investigate further, not for estimating true county prevalence.
- The next meaningful upgrade is replacing these synthetic profiles with Kenya-specific labelled clinical or survey data.